[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/vortices/vortices.ipynb)

# Leapfrogging Vortices

This example computes the motion of planar vortices and the spatial derivative of the flow they induce. Each vortex is modelled as a Rosenhead-Moore vortex blob that turns the surrounding fluid rapidly near its core and slower further out, while carrying the other vortices along with its induced velocity. Coaxial vortex pairs of opposite sense travel forward and take turns slipping through each other in a leapfrogging dance, whereas narrower inner pairs break symmetry into a temporary looping excursion called the walkabout before resuming the leapfrog. Traced upward through time, a regular ring of equal vortices winds into a uniform helix and the leapfrogging pairs braid around each other.

Each vortex carries an extrinsic circulation bivector whose magnitude gives its strength and whose sign gives the sense of turning, and a fluid parcel at separation `offset` moves with the commutator `offset.commutator(rate)`, so reflecting the plane reverses every circulation and mirrors the trajectory automatically. Differentiating with respect to displacement gives the closed-form velocity gradient `Point <- Point`, whose contraction `(Point * gradient(Point)).contract(1, 2)` combines scalar divergence and bivector vorticity in one even multivector, while `-0.5 * gradient(gradient).trace()` weighs swirl against strain without invoking a metric. Multiplying each vortex position on a regular ring by its initial position forms `ring_paths * ring_paths[0]`, a single time-varying rotor shared across every vortex on the ring.

In planar vector and complex Kirchhoff-Routh point-vortex notation, turning a displacement $\Delta\mathbf{x}_{ij} = \mathbf{x}_i - \mathbf{x}_j$ through a right angle embeds the plane in three dimensions via $\hat{\mathbf{z}} \times \Delta\mathbf{x}_{ij} / (2\pi r_{ij}^2)$, writes the component rule $\dot{x}_i = -\sum_j \frac{\Gamma_j (y_i - y_j)}{2\pi r_{ij}^2}$, $\dot{y}_i = \sum_j \frac{\Gamma_j (x_i - x_j)}{2\pi r_{ij}^2}$ with pseudoscalar signs $\Gamma_j$, or conjugates the complex velocity into $\overline{dz_i/dt} = \frac{1}{2\pi i} \sum_{j \neq i} \frac{\Gamma_j}{z_i - z_j}$, while separating the velocity gradient $\nabla\mathbf{v}$ into scalar divergence $\partial_x u + \partial_y v$, out-of-plane vorticity $(\partial_x v - \partial_y u)\hat{\mathbf{z}}$, and symmetric-skew matrices $\mathbf{S}, \boldsymbol{\Omega}$ for $Q = \tfrac{1}{2}(\lVert\boldsymbol{\Omega}\rVert^2 - \lVert\mathbf{S}\rVert^2)$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA2D as ga
from examples.util.animation import show_states
from examples.fluids.vortices import render, scenarios

np.set_printoptions(precision=3, suppress=True)

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                  # Scalar+
# Points, displacements, and velocities are antivectors with intrinsic orientation.
Point = ga.gatype.antivector()                               # Antivector-
# Circulation and fluid turning rates are bivectors with extrinsic orientation.
Circulation = ga.gatype.bivector()                           # Bivector+
Rate = ga.gatype.bivector()                                  # Bivector+
Even = ga.gatype.even()                                      # Even+
Gradient = ga.gatype((Point, Point))                         # Point- <- Point-

## Induced flow around vortex blobs

Each vortex turns the fluid around its centre with an extrinsic circulation bivector `xy * strengths` that records both strength and sense of rotation. Turning a separation `offsets` through a quarter turn by the local turning rate `rates` is the commutator `offsets.commutator(rates)`, and summing over the vortices gives the fluid velocity at each sample point. Under the Rosenhead-Moore blob profile, the core turns like a solid disc near the centre while falling off inversely with distance outside.

In [ ]:
# Two coaxial counter-rotating pairs along y so that both pairs travel along +x.
half_spans = np.array([1.0, -1.0, 0.55, -0.55])
strengths = np.array([1.0, -1.0, 1.0, -1.0])
core_radius = 0.15
# An isolated group carries a single zero-offset periodic copy.
isolated = mv.vector([[0, 0]]).dual()                                         # [1] Point-
half_width, half_height = 2.4, 1.6
n_columns, n_rows = 150, 100
n_samples_per_side = 2
n_sample_columns = n_columns * n_samples_per_side
n_sample_rows = n_rows * n_samples_per_side

In [ ]:
def separations(centres: Point, copies: Point, points: Point) -> Point:
    """The displacements from every vortex and periodic copy to every evaluation point."""
    return points[..., None, None] - (centres[..., None] + copies)            # [...] Point-


def blob_profile(circulations: Circulation, core_radius: float, offsets: Point) -> tuple[Rate, Rate]:
    """The Rosenhead-Moore turning rate and its derivative with respect to squared distance."""
    blob = (offsets | offsets) + core_radius**2                               # [...] Scalar+
    rates = circulations[:, None] / (2 * np.pi) / blob                        # [...] Rate+
    return rates, -rates / blob

In [ ]:
def velocity(centres: Point, circulations: Circulation, core_radius: float, copies: Point, points: Point) -> Point:
    """The fluid velocity induced at each point by the vortices and their periodic copies."""
    offsets = separations(centres, copies, points)                            # [...] Point-
    rates, _ = blob_profile(circulations, core_radius, offsets)               # [...] Rate+
    # Each vortex moves the fluid along the separation turned a quarter turn by its local rate.
    return offsets.commutator(rates).sum(axis=-1).sum(axis=-1)                # [...] Point-


# Embedding axis points into full planar Point widens their layout before stepping.
centres = (mv.y * half_spans).dual().cast(Point)                              # [4] Point-
circulations = mv.xy * strengths                                              # [4] Circulation+
sample_points = centres.mean(axis=-1) + scenarios.grid(half_width, half_height, n_sample_columns, n_sample_rows)   # [n_sample_rows, n_sample_columns] Point-
flow_velocity = velocity(centres, circulations, core_radius, isolated, sample_points)                     # [n_sample_rows, n_sample_columns] Point-

## Velocity gradient and flow derivative

The velocity gradient at a point is a linear map `Point <- Point` taking a small displacement to the resulting change in fluid velocity. Displacing the evaluation point turns the step with the local rate via `Point.commutator(rates)` and changes the rate through the radial projection `offsets | Point`. Multiplying the map's output by `Point` and contracting across input and output forms `(Point * gradients(Point)).contract(1, 2)`, the geometric derivative of the flow. Its scalar part is the divergence, which vanishes identically for incompressible vortex blobs, and its bivector part is the vorticity concentrated inside the cores. Away from the cores the vorticity decays as the fourth power of distance, approaching an analytic potential flow.

In complex-analysis notation, writing the planar velocity as $w = u - iv$ expresses vanishing divergence and vorticity away from the cores as the Cauchy-Riemann equations $\partial_x u + \partial_y v = 0$ and $\partial_y u - \partial_x v = 0$.

In [ ]:
def gradient(centres: Point, circulations: Circulation, core_radius: float, copies: Point, points: Point) -> Gradient:
    """The spatial velocity gradient mapping a small displacement to the resulting velocity change."""
    offsets = separations(centres, copies, points)                            # [...] Point-
    rates, rate_slopes = blob_profile(circulations, core_radius, offsets)     # [...] Rate+, Rate+
    # A displacement turns with the local rate and shifts the rate through its radial component.
    turning = Point.commutator(rates)                                         # [...] Point- <- Point-
    radial_shear = 2 * offsets.commutator(rate_slopes) * (offsets | Point)    # [...] Point- <- Point-
    return (turning + radial_shear).sum(axis=-1).sum(axis=-1)                 # [...] Point- <- Point-


def derivative(gradients: Gradient) -> Even:
    """The geometric derivative of the flow, combining divergence and vorticity."""
    return (Point * gradients(Point)).contract(1, 2)                          # [...] Even+


gradients = gradient(centres, circulations, core_radius, isolated, sample_points)   # [n_sample_rows, n_sample_columns] Point- <- Point-
derivatives = derivative(gradients)                                                 # [n_sample_rows, n_sample_columns] Even+
midpoint = (centres[0] + centres[2]) / 2                                            # [] Point-
midpoint_gradient = gradient(centres, circulations, core_radius, isolated, midpoint)   # [] Point- <- Point-
midpoint_derivative = derivative(midpoint_gradient)                                 # [] Even+

At the midpoint between the two upper vortices, the velocity gradient has unequal off-diagonal entries whose difference appears as the bivector vorticity in the flow derivative while the scalar divergence vanishes.

In [ ]:
print(f"gradient\n{midpoint_gradient.coefficient_table()}\n")
print(f"derivative\n{midpoint_derivative.coefficient_table()}")

## Swirl against strain

The antisymmetric part of the velocity gradient turns fluid parcels while the symmetric part stretches them. Composing the gradient with itself and taking half the negated trace, `-0.5 * gradients(gradients).trace()`, pairs output directly with input without a metric and measures the excess of turning squared over stretching squared. For a divergence-free planar flow this invariant equals the determinant of the velocity gradient, taking positive values inside vortex cores and negative values in the saddle regions where neighbouring vortices pull fluid apart.

In continuum-mechanics tensor notation, decomposing $\nabla\mathbf{v} = \mathbf{S} + \boldsymbol{\Omega}$ into symmetric strain rate $\mathbf{S}$ and skew spin $\boldsymbol{\Omega}$ writes this invariant as the Okubo-Weiss or $Q$-criterion $Q = \tfrac{1}{2}(\lVert\boldsymbol{\Omega}\rVert^2 - \lVert\mathbf{S}\rVert^2)$.

In [ ]:
vorticity_scale, swirl_scale = 14.0, 6.0

In [ ]:
def swirl(gradients: Gradient) -> Scalar:
    """The excess of turning squared over strain squared, positive inside vortex cores."""
    return -0.5 * gradients(gradients).trace()                                # [...] Scalar+


swirls = swirl(gradients)                                                     # [n_sample_rows, n_sample_columns] Scalar+
midpoint_swirl = swirl(midpoint_gradient)                                     # [] Scalar+
render.figure(sample_points, derivatives, swirls, vorticity_scale, swirl_scale, n_samples_per_side);

In the left panel, red marks counterclockwise vorticity and blue marks clockwise vorticity on a white calm background, deepening where cores overlap. In the right panel, orange marks swirl-dominated vortex cores and purple marks the strain-dominated saddle regions between them.

## Leapfrogging pairs

Each vortex drifts with the velocity induced by the other vortices at its centre, where its own blob contributes zero, integrated here with the classical fourth-order Runge-Kutta step. Because the narrower inner pair induces a stronger mutual velocity, it slips ahead through the outer pair, widening and slowing as the outer pair contracts, accelerates, and overtakes in turn.

In [ ]:
dt = 0.12
frame_duration_ms = 40
n_frames, n_substeps = 120, 2
pixels_per_unit = 40
trip_centre = mv.x.dual() * 3.0                                               # [] Point-
trip_half_width, trip_half_height = 3.6, 1.4

In [ ]:
def drift(centres: Point, circulations: Circulation, core_radius: float, copies: Point) -> Point:
    """The induced velocity of each vortex at its own centre."""
    return velocity(centres, circulations, core_radius, copies, centres)      # [...] Point-


def step(centres: Point, circulations: Circulation, core_radius: float, copies: Point, dt: float) -> Point:
    """The vortex centres advanced over one time step `dt` by fourth-order Runge-Kutta integration."""
    first = drift(centres, circulations, core_radius, copies)                 # [...] Point-
    second = drift(centres + first * (0.5 * dt), circulations, core_radius, copies)   # [...] Point-
    third = drift(centres + second * (0.5 * dt), circulations, core_radius, copies)   # [...] Point-
    fourth = drift(centres + third * dt, circulations, core_radius, copies)           # [...] Point-
    return centres + (first + 2 * second + 2 * third + fourth) * (dt / 6)     # [...] Point-

In [ ]:
def trajectory(
    centres: Point,
    circulations: Circulation,
    core_radius: float,
    copies: Point,
    n_frames: int,
    n_substeps: int,
    dt: float,
) -> Iterator[Point]:
    """Yield the vortex centres across `n_frames` frames separated by `n_substeps` steps."""
    for _ in range(n_frames):
        yield centres
        for _ in range(n_substeps):
            centres = step(centres, circulations, core_radius, copies, dt)


trip_shape = (round(2 * trip_half_height * pixels_per_unit), round(2 * trip_half_width * pixels_per_unit))
leapfrog_states = show_states(
    trajectory(centres, circulations, core_radius, isolated, n_frames, n_substeps, dt),
    render.window,
    circulations,
    core_radius,
    isolated,
    trip_centre,
    trip_half_height,
    trip_shape,
    vorticity_scale,
    name="vortices_leapfrog",
    duration_ms=frame_duration_ms,
    n_frames=n_frames,
)

## Symmetry-breaking walkabout

Coaxial leapfrogging remains stable only while the inner span exceeds the reciprocal squared golden ratio, roughly 0.38 of the outer span, as established by [Behring and Goodman](https://arxiv.org/abs/2210.16464). Between ratios of about 0.29 and 0.38, a small longitudinal nudge triggers the walkabout ([Behring and Goodman, 2019](https://arxiv.org/abs/1908.08618)), in which the two vortices of one circulation sense bind into a tight pair encircled by the other two until the group reopens and resumes leapfrogging.

In [ ]:
walkabout_spans = np.array([1.0, -1.0, 0.35, -0.35])
walkabout_nudges = np.array([0.0, 0.0, 0.015, 0.0])
walkabout_dt = 0.2
n_walkabout_frames, n_walkabout_substeps = 135, 5
walkabout_centre = (mv.x * 10.0 + mv.y * 1.4).dual()                          # [] Point-
walkabout_half_width, walkabout_half_height = 10.6, 2.9
walkabout_pixels_per_unit = 20

In [ ]:
walkabout_start = (mv.y * walkabout_spans + mv.x * walkabout_nudges).dual()   # [4] Point-
walkabout_shape = (
    round(2 * walkabout_half_height * walkabout_pixels_per_unit),
    round(2 * walkabout_half_width * walkabout_pixels_per_unit),
)
walkabout_states = show_states(
    trajectory(
        walkabout_start,
        circulations,
        core_radius,
        isolated,
        n_walkabout_frames,
        n_walkabout_substeps,
        walkabout_dt,
    ),
    render.window,
    circulations,
    core_radius,
    isolated,
    walkabout_centre,
    walkabout_half_height,
    walkabout_shape,
    vorticity_scale,
    name="vortices_walkabout",
    duration_ms=frame_duration_ms,
    n_frames=n_walkabout_frames,
)

## Worldline braids

Stacking the vortex centres along a vertical time axis traces each vortex's worldline in the comoving frame. Five equal vortices arranged on a regular ring by even rotors `rotors >> initial_point` rotate rigidly around their common centre, winding five helices around one vertical axis, and the geometric product `ring_paths * ring_paths[0]` yields the same scaled rotor for all five vortices at every step. By contrast, the two like-signed vortices on each flank of a leapfrogging pair wind around each other once every two passes, braiding their worldlines.

In [ ]:
n_ring, ring_radius, ring_core = 5, 1.0, 0.05
ring_strengths = np.ones(n_ring)
worldline_dt = 0.2
n_ring_steps, n_leapfrog_steps = 150, 75
braid_height = 2.5

In [ ]:
def worldlines(
    centres: Point,
    circulations: Circulation,
    core_radius: float,
    copies: Point,
    dt: float,
    n_steps: int,
) -> Point:
    """Worldline positions across `n_steps + 1` instants for each vortex."""
    return stack(list(trajectory(centres, circulations, core_radius, copies, n_steps + 1, 1, dt)))   # [n_steps + 1, ...] Point-


rotors = (mv.xy * (-np.pi * np.arange(n_ring) / n_ring)).exp()                # [n_ring] Even+
ring_centres = rotors >> (mv.x.dual() * ring_radius)                          # [n_ring] Point-
ring_circulations = mv.xy * ring_strengths                                    # [n_ring] Circulation+
ring_paths = worldlines(ring_centres, ring_circulations, ring_core, isolated, worldline_dt, n_ring_steps)   # [n_ring_steps + 1, n_ring] Point-
pair_paths = worldlines(centres, circulations, core_radius, isolated, worldline_dt, n_leapfrog_steps)       # [n_leapfrog_steps + 1, 4] Point-
render.braids([ring_paths, pair_paths], braid_height);

Each coloured tube traces one vortex upward from its initial planar position in the comoving frame, with vertical height scaled to a fixed multiple of the pattern's horizontal span.

In [ ]:
# checks
# Reflecting the coaxial pairs across the x-axis swaps each opposite-signed pair and mirrors the flow.
assert (((mv.y >> flow_velocity) - flow_velocity[::-1]).norm() < 1e-10).all()
# The flow derivative has zero divergence and its bivector part equals the closed-form blob vorticity.
offsets = separations(centres, isolated, sample_points)                       # [...] Point-
blob = (offsets | offsets) + core_radius**2                                   # [...] Scalar+
vorticity = (circulations[:, None] * core_radius**2 / np.pi / (blob * blob)).sum(axis=-1).sum(axis=-1)   # [n_sample_rows, n_sample_columns] Rate+
assert ((derivatives - vorticity).norm() < 1e-10).all()
assert (midpoint_derivative.cast(Scalar).abs() < 1e-10).all()
# Without divergence, planar swirl equals the gradient determinant and is negative in the saddle between upper cores.
assert ((swirls - gradients.det()).abs() < 1e-10).all()
assert (midpoint_swirl < 0).all()
# Hydrodynamic linear and angular impulse stay conserved across both single-run trajectories.
for run_states in (stack(leapfrog_states), stack(walkabout_states)):
    linear_impulse = (run_states * circulations).sum(axis=-1)                 # [...] Point-
    angular_impulse = (circulations * (run_states | run_states)).sum(axis=-1) # [...] Circulation+
    assert ((linear_impulse - linear_impulse[0]).norm() < 1e-9).all()
    assert ((angular_impulse - angular_impulse[0]).norm() < 1e-4).all()
# Impulse stays conserved on both worldline braids, and all five ring vortices share one time-varying rotor.
for braid_circulations, paths in ((ring_circulations, ring_paths), (circulations, pair_paths)):
    linear_impulse = (paths * braid_circulations).sum(axis=-1)                # [...] Point-
    assert ((linear_impulse - linear_impulse[0]).norm() < 1e-9).all()
relative_rotors = ring_paths * ring_paths[0]                                  # [n_ring_steps + 1, n_ring] Even+
assert ((relative_rotors - relative_rotors[:, :1]).norm() < 1e-9).all()